In [1]:
from fredapi import Fred
import pandas as pd
from datetime import datetime, timedelta

In [2]:
fred = Fred(api_key='f58608dfefdfb2fb544373a205734834')

start_date = (datetime.today() - timedelta(days=500)).strftime("%Y-%m-%d")


series = {
    # Fed Policy
    "fedfunds_overnight_bankTobank": "FEDFUNDS",

    # Inflation
    "pce": "PCEPI",
    "core_pce": "PCEPILFE",
    "core_cpi": "CPILFESL",
    "cpi": "CPIAUCSL",

    # Treasury Constant Maturity Rates
    "treasury_5y_constant_maturity_interpolation": "DGS5",
    "treasury_7y_constant_maturity_interpolation": "DGS7",
    "treasury_10y_constant_maturity_interpolation": "DGS10",
    "treasury_20y_constant_maturity_interpolation": "DGS20",
    "treasury_30y_constant_maturity_interpolation": "DGS30",

    # Breakeven Inflation
    "breakeven_5y_nominal_tips": "T5YIE",
    "breakeven_7y_nominal_tips": "T7YIEM",
    "breakeven_10y_nominal_tips": "T10YIE",
    "breakeven_20y_nominal_tips": "T20YIEM",
    "breakeven_30y_nominal_tips": "T30YIEM",

    # Corporate Bond Yields
    "baa_corp": "BAA",
    "aaa_corp": "AAA",
    
    # jobs
    "Indeed job postings": "IHLIDXUS",
    "Job_openings": "JTSJOL",
    "labor force participation rate (percentage)": "LNS11300060",
    "hours worked": "AVHWPEUSA065NRUG",
    "labor_force_participation": "CIVPART",

    
    #job less
    "intitial jobless claims": "ICSA",
    "unemployment_level": "UNEMPLOY",
    "long term unemployed": "UEMP27OV",
    "median weeks unemployed": "UEMPMED",
    "unemployment_rate": "UNRATE",
    "layoffs versus job openings plus employment": "JTSLDR",
    
    #wages
    "real earnings weekly (real)": "LES1252881600Q",
}


data_last_500_days = pd.DataFrame({
    name: fred.get_series(series_id, observation_start=start_date)
    for name, series_id in series.items()
})

data_last_500_days.tail(5)

/Users/nicholassanso/anaconda3/envs/dsi/lib/python3.8/site-packages/fredapi/fred.py:162: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  return pd.Series(data)


,fedfunds_overnight_bankTobank,pce,core_pce,core_cpi,cpi,treasury_5y_constant_maturity_interpolation,treasury_7y_constant_maturity_interpolation,treasury_10y_constant_maturity_interpolation,treasury_20y_constant_maturity_interpolation,treasury_30y_constant_maturity_interpolation,...,labor force participation rate (percentage),hours worked,labor_force_participation,intitial jobless claims,unemployment_level,long term unemployed,median weeks unemployed,unemployment_rate,layoffs versus job openings plus employment,real earnings weekly (real)
2026-06-20 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,216000.0,NaN,NaN,NaN,NaN,NaN,NaN
2026-06-21 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-06-27 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,217000.0,NaN,NaN,NaN,NaN,NaN,NaN
2026-07-04 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,215000.0,NaN,NaN,NaN,NaN,NaN,NaN
2025-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,373.0


In [5]:
import pandas as pd
import plotly.graph_objects as go
from fredapi import Fred
from datetime import datetime, timedelta

# 1. Pull FRED data
data_last_500_days = pd.DataFrame({
    name: fred.get_series(series_id, observation_start=start_date)
    for name, series_id in series.items()
})

data_last_500_days.index = pd.to_datetime(data_last_500_days.index)

# 2. Forward-fill so monthly/weekly/quarterly values show latest available value
dashboard_data = data_last_500_days.sort_index().ffill()

# 3. Get latest available values
latest = dashboard_data.iloc[-1]

# 4. Build yield curve dashboard table
maturities = [5, 7, 10, 20, 30]

curve_df = pd.DataFrame({
    "Maturity": maturities,

    "Treasury Yield": [
        latest["treasury_5y_constant_maturity_interpolation"],
        latest["treasury_7y_constant_maturity_interpolation"],
        latest["treasury_10y_constant_maturity_interpolation"],
        latest["treasury_20y_constant_maturity_interpolation"],
        latest["treasury_30y_constant_maturity_interpolation"],
    ],

    "Breakeven Inflation": [
        latest["breakeven_5y_nominal_tips"],
        latest["breakeven_7y_nominal_tips"],
        latest["breakeven_10y_nominal_tips"],
        latest["breakeven_20y_nominal_tips"],
        latest["breakeven_30y_nominal_tips"],
    ],
})

# 5. Back out TIPS real yield
curve_df["TIPS Real Yield"] = (
    curve_df["Treasury Yield"] - curve_df["Breakeven Inflation"]
)

# 6. Plot all 3 lines
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=curve_df["Maturity"],
    y=curve_df["Treasury Yield"],
    mode="lines+markers",
    name="Treasury Yield"
))

fig.add_trace(go.Scatter(
    x=curve_df["Maturity"],
    y=curve_df["TIPS Real Yield"],
    mode="lines+markers",
    name="TIPS Real Yield"
))

fig.add_trace(go.Scatter(
    x=curve_df["Maturity"],
    y=curve_df["Breakeven Inflation"],
    mode="lines+markers",
    name="Breakeven Inflation"
))

fig.update_layout(
    title="Treasury Yield Curve, TIPS Real Yield Curve, and Breakeven Inflation",
    xaxis_title="Maturity",
    yaxis_title="Rate (%)",
    template="plotly_white",
    hovermode="x unified",
    width=950,
    height=550
)

fig.update_xaxes(
    tickmode="array",
    tickvals=maturities,
    ticktext=["5Y", "7Y", "10Y", "20Y", "30Y"]
)

fig.show()

curve_df

ValueError: Bad Request.  The value for variable api_key is not a 32 character alpha-numeric lower-case string.  Read https://fred.stlouisfed.org/docs/api/api_key.html for more information.